# Local forecasting workbench
Train and compare models through the DI Validator worker. Python here runs in JupyterLite; model fitting runs in the local workstation environments. No Hugging Face models or checkpoints are used.

This example reads the supplied anomaly CSV without inferring its timezone or treating its labels as verified evidence. Forecasting is independent of anomaly classification. Change the source, channel, horizon, and models to suit your analysis.

In [ ]:
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import di_forecast as forecasts
catalog = await forecasts.models()
pd.DataFrame(catalog)[['id', 'name', 'ready', 'version']]

## Submit a chronological benchmark
The worker fits each model on observations before each validation window. The final 12 observations are held out from model selection. A separate full-history fit produces future forecasts. Ensembles use equal weights. Missing data is rejected by default.

In [ ]:
configuration = dict(
    name="Notebook - power anomaly forecast",
    source_file="power_anomaly_dataset_2022.csv",
    timestamp_column="Timestamp", channel="Load_kW", unit="unverified",
    horizon=12, validation_windows=2, context_length=256, max_history=1000,
    season_length=24, lags=24, seed=42,
    models=["seasonal_naive", "random_forest", "ensemble"],
    parameters={"random_forest": {"n_estimators":30}},
)
submitted = await forecasts.start(configuration)
print("Job:", submitted["id"])
for _ in range(300):
    status = await forecasts.job(submitted["id"])
    if status["status"] not in ("queued", "running"):
        break
    await asyncio.sleep(1)
if status["status"] != "completed":
    raise RuntimeError(status.get("error") or status.get("message"))
run = next(r for r in await forecasts.runs() if r["id"] == status["result"]["id"])
print("Completed:", run["id"], "Selected:", run["selected_model"])
pd.DataFrame([dict(model=m["model"], **m.get("holdout", {})) for m in run["models"]])

In [ ]:
predictions = await forecasts.predictions(run["id"], run["selected_model"])
holdout = predictions[predictions.phase == "holdout"].copy()
holdout["timestamp"] = pd.to_datetime(holdout.timestamp)
holdout.set_index("timestamp")[["actual", "prediction"]].plot(
    title="Final holdout - reported source clock; units unverified", figsize=(10, 3))
print("Export:", f"/api/v1/forecasting/runs/{run['id']}/export")
plt.show()

## Use indexed measurements or saved fitted models
For indexed data, replace `source_file` with `dataset_id`, `asset_id`, and a declared `channel`. Use `di_data.DIClient` to browse these identities. Optional `start` and `end` bound the history. Units and provenance then come from the imported dataset.

To reuse a fitted model, copy a completed backtest configuration, set `reuse_run_id` and `reuse_model` to an individual successful model, and submit with `forecasts.start`. The worker checks the saved environment, feature schema, source identity, artifact checksums, and training cutoff. It does not retrain or report evaluation metrics for inference runs. Re-run the benchmark when new fitting is desired.

Use `await forecasts.cancel(job_id)` to cancel a queued/running job. Notebook interruption alone does not cancel its workstation job. Download notebooks from JupyterLite to back up custom scripts.